### Passo 1 - Importar Arquivos e Bibliotecas

In [1]:
import pandas as pd
import pathlib
import os
import smtplib
from dotenv import load_dotenv

emails = pd.read_excel("Bases de Dados/Emails.xlsx")
lojas = pd.read_csv("Bases de Dados/Lojas.csv", encoding="latin1", sep=";")
vendas = pd.read_excel("Bases de Dados/Vendas.xlsx")

### Passo 2 - Criar uma Tabela para cada Loja e Definir o dia do Indicador

In [2]:
vendas = vendas.merge(lojas, on="ID Loja")

In [3]:
dicionario_lojas = {}
for loja in lojas["Loja"]:
    dicionario_lojas[loja] = vendas.loc[vendas["Loja"]==loja, :]

In [4]:
dia_indicador = vendas["Data"].max()

### Passo 3 - Salvar a planilha na pasta de backup

In [5]:
caminho_backup = pathlib.Path(r"Backup Arquivos Lojas")
arquivos_pasta_backup = caminho_backup.iterdir()

lista_nomes_backup = [arquivo.name for arquivo in arquivos_pasta_backup]

for loja in dicionario_lojas:
    if loja not in lista_nomes_backup:
        nova_pasta = caminho_backup / loja
        nova_pasta.mkdir()

    nome_arquivo = f"{dia_indicador.month}_{dia_indicador.day}_{loja}.xlsx"
    local_arquivo = caminho_backup / loja / nome_arquivo

    dicionario_lojas[loja].to_excel(local_arquivo)

### Passo 4 - Calcular os indicadores e enviar os e-mails

In [6]:
#Definição de Metas

meta_faturamento_dia = 1000
meta_faturamento_ano = 1650000
meta_qtde_produtos_dia = 4
meta_qtde_produtos_ano = 120
meta_ticket_medio_dia = 500
meta_ticket_medio_ano = 500

In [7]:
for loja in dicionario_lojas:
    vendas_loja = dicionario_lojas[loja]
    vendas_loja_dia = vendas_loja.loc[vendas_loja["Data"]==dia_indicador, :]

    #faturamento
    faturamento_ano = vendas_loja["Valor Final"].sum(numeric_only=True)
    faturamento_dia = vendas_loja_dia["Valor Final"].sum(numeric_only=True)

    #diversidade de produtos
    qtde_produtos_ano = len(vendas_loja["Produto"].unique())
    qtde_produtos_dia = len(vendas_loja_dia["Produto"].unique())

    #ticket médio
    valor_venda = vendas_loja.groupby("Código Venda").sum(numeric_only=True)
    ticket_medio_ano = valor_venda["Valor Final"].mean()

    valor_venda_dia = vendas_loja_dia.groupby("Código Venda").sum(numeric_only=True)
    ticket_medio_dia = valor_venda_dia["Valor Final"].mean()

    from email.message import EmailMessage

    nome = emails.loc[emails["Loja"] == loja, "Gerente"].values[0]

    msg = EmailMessage()

    load_dotenv()

    meu_email = os.getenv("MEU_EMAIL")

    msg["From"] = meu_email
    msg["To"] = emails.loc[emails["Loja"] == loja, "E-mail"].values[0]
    msg["Cc"] = meu_email

    msg["Subject"] = (
        f"OnePage Dia {dia_indicador.day}/{dia_indicador.month} - Loja {loja}"
    )

    if faturamento_dia >= meta_faturamento_dia:
        cor_fat_dia = "green"
    else:
        cor_fat_dia = "red"

    if faturamento_ano >= meta_faturamento_ano:
        cor_fat_ano = "green"
    else:
        cor_fat_ano = "red"

    if qtde_produtos_dia >= meta_qtde_produtos_dia:
        cor_qtde_dia = "green"
    else:
        cor_qtde_dia = "red"

    if qtde_produtos_ano >= meta_qtde_produtos_ano:
        cor_qtde_ano = "green"
    else:
        cor_qtde_ano = "red"

    if ticket_medio_dia >= meta_ticket_medio_dia:
        cor_ticket_dia = "green"
    else:
        cor_ticket_dia = "red"

    if meta_ticket_medio_ano >= meta_ticket_medio_dia:
        cor_ticket_ano = "green"
    else:
        cor_ticket_ano = "red"

    msg.set_content(
        f"Bom dia, {nome}\n\n"
        f"Segue o resultado da Loja {loja}.\n"
        f"Caso seu cliente de e-mail não suporte HTML, consulte o anexo."
    )

    msg.add_alternative(f"""
    <html>
    <body>
        <p>Bom dia, {nome}</p>

        <p>O resultado de ontem <strong>({dia_indicador.day}/{dia_indicador.month})</strong> da <strong>Loja {loja}</strong> foi:</p>

        <table border="1" cellpadding="5" cellspacing="0" style="border-collapse: collapse;">
        <tr>
            <th>Indicador</th>
            <th>Valor Dia</th>
            <th>Meta Dia</th>
            <th>Cenário Dia</th>
        </tr>
        <tr>
            <td>Faturamento</td>
            <td style="text-align:center">R${faturamento_dia:.2f}</td>
            <td style="text-align:center">R${meta_faturamento_dia:.2f}</td>
            <td style="text-align:center"><span style="color:{cor_fat_dia};">◙</span></td>
        </tr>
        <tr>
            <td>Diversidade de Produtos</td>
            <td style="text-align:center">{qtde_produtos_dia}</td>
            <td style="text-align:center">{meta_qtde_produtos_dia}</td>
            <td style="text-align:center"><span style="color:{cor_qtde_dia};">◙</span></td>
        </tr>
        <tr>
            <td>Ticket Médio</td>
            <td style="text-align:center">R${ticket_medio_dia:.2f}</td>
            <td style="text-align:center">R${meta_ticket_medio_dia:.2f}</td>
            <td style="text-align:center"><span style="color:{cor_ticket_dia};">◙</span></td>
        </tr>
        </table>

        <br>

        <table border="1" cellpadding="5" cellspacing="0" style="border-collapse: collapse;">
        <tr>
            <th>Indicador</th>
            <th>Valor Ano</th>
            <th>Meta Ano</th>
            <th>Cenário Ano</th>
        </tr>
        <tr>
            <td>Faturamento</td>
            <td style="text-align:center">R${faturamento_ano:.2f}</td>
            <td style="text-align:center">R${meta_faturamento_ano:.2f}</td>
            <td style="text-align:center"><span style="color:{cor_fat_ano};">◙</span></td>
        </tr>
        <tr>
            <td>Diversidade de Produtos</td>
            <td style="text-align:center">{qtde_produtos_ano}</td>
            <td style="text-align:center">{meta_qtde_produtos_ano}</td>
            <td style="text-align:center"><span style="color:{cor_qtde_ano};">◙</span></td>
        </tr>
        <tr>
            <td>Ticket Médio</td>
            <td style="text-align:center">R${ticket_medio_ano:.2f}</td>
            <td style="text-align:center">R${meta_ticket_medio_ano:.2f}</td>
            <td style="text-align:center"><span style="color:{cor_ticket_ano};">◙</span></td>
        </tr>
        </table>

        <p>Segue em anexo a planilha com todos os dados para mais detalhes.</p>

        <p>Qualquer dúvida estou à disposição.</p>

        <p>Att.,<br>Pedro</p>
    </body>
    </html>
    """, subtype="html")

    #anexo
    attachment = (pathlib.Path.cwd() / caminho_backup / loja / f"{dia_indicador.month}_{dia_indicador.day}_{loja}.xlsx")

    with open(attachment, "rb") as f:
        msg.add_attachment(
            f.read(),
            maintype="application",
            subtype="vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            filename=attachment.name
        )

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(
            os.getenv("MEU_EMAIL"),
            os.getenv("GMAIL_APP_PASSWORD")
        )
        smtp.send_message(msg)

### Passo 5 - Criar ranking para diretoria

In [8]:
faturamento_lojas_ano = vendas.groupby("Loja")[["Loja", "Valor Final"]].sum(numeric_only=True)
faturamento_lojas_ano = faturamento_lojas_ano.sort_values(by="Valor Final", ascending=False)

nome_arquivo = f"{dia_indicador.month}_{dia_indicador.day}_Ranking Anual.xlsx"
faturamento_lojas_ano.to_excel(rf"Backup Arquivos Lojas\{nome_arquivo}")

vendas_dia = vendas.loc[vendas["Data"]==dia_indicador, :]
faturamento_lojas_dia = vendas_dia.groupby("Loja")[["Loja", "Valor Final"]].sum(numeric_only=True)
faturamento_lojas_dia = faturamento_lojas_dia.sort_values(by="Valor Final", ascending=False)

nome_arquivo = f"{dia_indicador.month}_{dia_indicador.day}_Ranking Dia.xlsx"
faturamento_lojas_dia.to_excel(rf"Backup Arquivos Lojas\{nome_arquivo}")

### Passo 8 - Enviar e-mail para diretoria

In [9]:
from email.message import EmailMessage

msg = EmailMessage()

msg["From"] = meu_email
msg["To"] = emails.loc[emails["Loja"] == "Diretoria", "E-mail"].values[0]
msg["Cc"] = meu_email

msg["Subject"] = (
    f"Ranking Dia {dia_indicador.day}/{dia_indicador.month}"
)


msg.set_content(f"""
Bom dia!
                
Melhor loja do dia em faturamento: {faturamento_lojas_dia.index[0]} com faturamento R${faturamento_lojas_dia.iloc[0,0]:.2f}
Pior loja do dia em faturamento: {faturamento_lojas_dia.index[-1]} com faturamento R${faturamento_lojas_dia.iloc[-1,0]:.2f}

Melhor loja do ano em faturamento: {faturamento_lojas_ano.index[0]} com faturamento R${faturamento_lojas_ano.iloc[0,0]:.2f}
Pior loja do ano em faturamento: {faturamento_lojas_ano.index[-1]} com faturamento R${faturamento_lojas_ano.iloc[-1,0]:.2f}

Segue em anexo os rankings do ano e do dia de todas as lojas.

Qualquer dúvida, estou à disposição!

Att., Pedro

"""
    
)


#anexo
attachment = (pathlib.Path.cwd() / caminho_backup / f"{dia_indicador.month}_{dia_indicador.day}_Ranking Anual.xlsx")
with open(attachment, "rb") as f:
    msg.add_attachment(
        f.read(),
        maintype="application",
        subtype="vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        filename=attachment.name
    )

attachment = (pathlib.Path.cwd() / caminho_backup / f"{dia_indicador.month}_{dia_indicador.day}_Ranking Dia.xlsx")

with open(attachment, "rb") as f:
    msg.add_attachment(
        f.read(),
        maintype="application",
        subtype="vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        filename=attachment.name
    )

load_dotenv()

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
    smtp.login(
        os.getenv("MEU_EMAIL"),
        os.getenv("GMAIL_APP_PASSWORD")
    )
    smtp.send_message(msg)